In [1]:
# Block 0 — setup. Code from GitHub onto the VM disk; data + embeddings on Drive.
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

# numpy/scipy/torch are filtered out: Colab's torch is built against its own numpy
!grep -viE '^(torch|torchvision|torchaudio|nvidia-|numpy|scipy|#)' requirements.txt > req_colab.txt
!pip install -q -r req_colab.txt

Mounted at /content/drive
Cloning into '/content/NeoAntigen'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 145 (delta 83), reused 105 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 1.60 MiB | 4.72 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/content/NeoAntigen


In [2]:
# Block 1 — the WHOLE pan-cancer manifest (every peptide of every arm) and its embedding cache key.
# Every arm frame (colorectal / lung / melanoma / pooled, any fold) is a subset of the manifest, so the embeddings are computed
# ONCE here and every arm reads this cache. The key is a hash of the manifest's (uid, sequence) pairs.
import torch
from src import config, data, embed, pancancer_split as ps

NAME = "pancancer_cedar_v1"
manifest = ps.load_manifest(config.DATA_DIR, NAME)
EID = ps.emb_id(manifest)
print("DATA_DIR :", config.DATA_DIR)
print("cache key:", EID)
print("sequences:", len(manifest), "| residues:", int(manifest["length"].sum()))
print("GPU      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

DATA_DIR : /content/drive/MyDrive/s1156052/NeoAntigen/data
cache key: pc_d72e80deb5
sequences: 24663 | residues: 491662
GPU      : Tesla T4


In [3]:
# Block 2 — SMOKE TEST: tiny 8M checkpoint, a few seconds. Run once before committing to the 650M models.
embed.build_cache("esm2_8m", manifest, build_id=EID)
c = embed.EmbCache("esm2_8m", EID, staged=False)
x, lens = c.stack(manifest["uid"].head(4), int(manifest["length"].max()))
print("stack:", x.shape, x.dtype, "lengths:", lens, "| dim:", c.dim)

[esm2_8m] loading facebook/esm2_t6_8M_UR50D on cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 31.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[esm2_8m] shard 1/13  64/2048
[esm2_8m] shard 1/13  704/2048
[esm2_8m] shard 1/13  1344/2048
[esm2_8m] shard 1/13  1984/2048
[esm2_8m] shard 2/13  64/2048
[esm2_8m] shard 2/13  704/2048
[esm2_8m] shard 2/13  1344/2048
[esm2_8m] shard 2/13  1984/2048
[esm2_8m] shard 3/13  64/2048
[esm2_8m] shard 3/13  704/2048
[esm2_8m] shard 3/13  1344/2048
[esm2_8m] shard 3/13  1984/2048
[esm2_8m] shard 4/13  64/2048
[esm2_8m] shard 4/13  704/2048
[esm2_8m] shard 4/13  1344/2048
[esm2_8m] shard 4/13  1984/2048
[esm2_8m] shard 5/13  64/2048
[esm2_8m] shard 5/13  704/2048
[esm2_8m] shard 5/13  1344/2048
[esm2_8m] shard 5/13  1984/2048
[esm2_8m] shard 6/13  64/2048
[esm2_8m] shard 6/13  704/2048
[esm2_8m] shard 6/13  1344/2048
[esm2_8m] shard 6/13  1984/2048
[esm2_8m] shard 7/13  64/2048
[esm2_8m] shard 7/13  704/2048
[esm2_8m] shard 7/13  1344/2048
[esm2_8m] shard 7/13  1984/2048
[esm2_8m] shard 8/13  64/2048
[esm2_8m] shard 8/13  704/2048
[esm2_8m] shard 8/13  1344/2048
[esm2_8m] shard 8/13  1984/2048


In [4]:
# Block 3 — ProtTrans (ProtT5-XL, encoder-only fp16, 1024-d). The paper's primary feature set.
embed.build_cache("prott5", manifest, build_id=EID)

[prott5] loading Rostlab/prot_t5_xl_half_uniref50-enc on cuda


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.42GB            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.42GB            

model.safetensors: downloading bytes:           |  0.00B            

[prott5] shard 1/13  64/2048
[prott5] shard 1/13  704/2048
[prott5] shard 1/13  1344/2048
[prott5] shard 1/13  1984/2048
[prott5] shard 2/13  64/2048
[prott5] shard 2/13  704/2048
[prott5] shard 2/13  1344/2048
[prott5] shard 2/13  1984/2048
[prott5] shard 3/13  64/2048
[prott5] shard 3/13  704/2048
[prott5] shard 3/13  1344/2048
[prott5] shard 3/13  1984/2048
[prott5] shard 4/13  64/2048
[prott5] shard 4/13  704/2048
[prott5] shard 4/13  1344/2048
[prott5] shard 4/13  1984/2048
[prott5] shard 5/13  64/2048
[prott5] shard 5/13  704/2048
[prott5] shard 5/13  1344/2048
[prott5] shard 5/13  1984/2048
[prott5] shard 6/13  64/2048
[prott5] shard 6/13  704/2048
[prott5] shard 6/13  1344/2048
[prott5] shard 6/13  1984/2048
[prott5] shard 7/13  64/2048
[prott5] shard 7/13  704/2048
[prott5] shard 7/13  1344/2048
[prott5] shard 7/13  1984/2048
[prott5] shard 8/13  64/2048
[prott5] shard 8/13  704/2048
[prott5] shard 8/13  1344/2048
[prott5] shard 8/13  1984/2048
[prott5] shard 9/13  64/2048
[pr

PosixPath('/content/drive/MyDrive/s1156052/NeoAntigen/data/emb/pc_d72e80deb5/prott5')

In [6]:
# Block 4 — ESM-2 650M (1280-d)
embed.build_cache("esm2_650m", manifest, build_id=EID)

[esm2_650m] loading facebook/esm2_t33_650M_UR50D on cuda


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.61GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[esm2_650m] shard 1/13  64/2048
[esm2_650m] shard 1/13  704/2048
[esm2_650m] shard 1/13  1344/2048
[esm2_650m] shard 1/13  1984/2048
[esm2_650m] shard 2/13  64/2048
[esm2_650m] shard 2/13  704/2048
[esm2_650m] shard 2/13  1344/2048
[esm2_650m] shard 2/13  1984/2048
[esm2_650m] shard 3/13  64/2048
[esm2_650m] shard 3/13  704/2048
[esm2_650m] shard 3/13  1344/2048
[esm2_650m] shard 3/13  1984/2048
[esm2_650m] shard 4/13  64/2048
[esm2_650m] shard 4/13  704/2048
[esm2_650m] shard 4/13  1344/2048
[esm2_650m] shard 4/13  1984/2048
[esm2_650m] shard 5/13  64/2048
[esm2_650m] shard 5/13  704/2048
[esm2_650m] shard 5/13  1344/2048
[esm2_650m] shard 5/13  1984/2048
[esm2_650m] shard 6/13  64/2048
[esm2_650m] shard 6/13  704/2048
[esm2_650m] shard 6/13  1344/2048
[esm2_650m] shard 6/13  1984/2048
[esm2_650m] shard 7/13  64/2048
[esm2_650m] shard 7/13  704/2048
[esm2_650m] shard 7/13  1344/2048
[esm2_650m] shard 7/13  1984/2048
[esm2_650m] shard 8/13  64/2048
[esm2_650m] shard 8/13  704/2048
[esm

PosixPath('/content/drive/MyDrive/s1156052/NeoAntigen/data/emb/pc_d72e80deb5/esm2_650m')

In [5]:
# Block 5 — ESM-1b (1280-d)
embed.build_cache("esm1b", manifest, build_id=EID)

[esm1b] loading facebook/esm1b_t33_650M_UR50S on cuda


config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.61GB            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.61GB            

model.safetensors: downloading bytes:           |  0.00B            

[esm1b] shard 1/13  64/2048
[esm1b] shard 1/13  704/2048
[esm1b] shard 1/13  1344/2048
[esm1b] shard 1/13  1984/2048
[esm1b] shard 2/13  64/2048
[esm1b] shard 2/13  704/2048
[esm1b] shard 2/13  1344/2048
[esm1b] shard 2/13  1984/2048
[esm1b] shard 3/13  64/2048
[esm1b] shard 3/13  704/2048
[esm1b] shard 3/13  1344/2048
[esm1b] shard 3/13  1984/2048
[esm1b] shard 4/13  64/2048
[esm1b] shard 4/13  704/2048
[esm1b] shard 4/13  1344/2048
[esm1b] shard 4/13  1984/2048
[esm1b] shard 5/13  64/2048
[esm1b] shard 5/13  704/2048
[esm1b] shard 5/13  1344/2048
[esm1b] shard 5/13  1984/2048
[esm1b] shard 6/13  64/2048
[esm1b] shard 6/13  704/2048
[esm1b] shard 6/13  1344/2048
[esm1b] shard 6/13  1984/2048
[esm1b] shard 7/13  64/2048
[esm1b] shard 7/13  704/2048
[esm1b] shard 7/13  1344/2048
[esm1b] shard 7/13  1984/2048
[esm1b] shard 8/13  64/2048
[esm1b] shard 8/13  704/2048
[esm1b] shard 8/13  1344/2048
[esm1b] shard 8/13  1984/2048
[esm1b] shard 9/13  64/2048
[esm1b] shard 9/13  704/2048
[esm1b]

PosixPath('/content/drive/MyDrive/s1156052/NeoAntigen/data/emb/pc_d72e80deb5/esm1b')

In [7]:
# Block 6 — what is cached
for plm in ["prott5", "esm2_650m", "esm1b", "esm2_8m"]:
    ok = embed.is_cached(plm, EID)
    print(f"{plm:12s} {'cached' if ok else 'MISSING'}")

prott5       cached
esm2_650m    cached
esm1b        cached
esm2_8m      cached
